# Hybrid real-option counts (Kaggle CPU, internet ON, no GPU)
Exploratory (no gate). Counts how many states have REAL wrong options per evidence tier (T1 failing run + error/revert/retry, T2 failing run only, T3 real sibling), from the full nebius and SWE-smith `tool` trajectories, streamed inside Kaggle. Nothing is uploaded from the local machine; no Qwen, no states.
Logic: `tools/hybrid_options.py` (embedded below, same file as in the repo). Output: counts.json, rows_hybrid.jsonl.gz (option pools with tier, provenance, executed_here, soft_target; no state text yet).

In [ ]:
SRC = '"""Evidence-tiered REAL option pools per state (design: PLAN.md section 15). CPU only, stdlib only; nothing is executed or invented.\nA "state" is a shared action-prefix of several runs of the SAME instance (observations are not compared: approximation). At each such state:\n  GOLD  the real action of a run that resolved the task from this state\n  T1    real action of a failing-only branch, executed from this state, followed by an error / revert / same-command retry (weak evidence; an error after an action does not prove it caused the failure)\n  T2    real action of a failing-only branch, executed from this state, with no such sign\n  T3    real sibling, outcome unknown, never called wrong: another resolved branch\'s action, or an action from nearby steps of the gold run (executed_here=False)\n  NONE  always present, never a label\n`soft_target` is a fixed evidence prior (TIER_W, not tuned, not an outcome). Provenance and tier are stored per option; executed_here is True only for branches that really ran from this state.\n    python tools/hybrid_options.py selftest | counts SAMPLE.jsonl\n"""\nimport hashlib, json, re, sys\nfrom collections import Counter, defaultdict\n\nTIER_W = {"GOLD": 1.0, "T1": 0.02, "T2": 0.10, "T3": 0.25}\nACT_CHARS = 400\nERR = re.compile(r"exit code (?!0\\b)\\d+|Traceback \\(most recent|\\bError\\b|FAILED|command not found|No such file")\nREV = re.compile(r"git (checkout|stash|reset|restore)|undo_edit")\nPATH = re.compile(r"[\\w./-]+\\.(?:py|js|ts|md|txt|toml|cfg|json|ya?ml|rs|go|java|c|h|cpp)")\n\n\ndef norm(a):\n    t = a.split()\n    p = PATH.findall(a)\n    return (t[0] if t else "", t[1] if len(t) > 1 and not t[1].startswith("{") else "", p[0].split("/")[-1] if p else "")\n\n\ndef _jl(x):\n    try:\n        return json.loads(x) if isinstance(x, str) else x\n    except Exception:\n        return None\n\n\nFUNC = re.compile(r"<function=([\\w.-]+)>(.*?)</function>", re.S)\nPARAM = re.compile(r"<parameter=([\\w.-]+)>(.*?)</parameter>", re.S)\n\n\ndef _xml_act(c):\n    """SWE-smith rows without `action` carry the call inside the message text as <function=NAME><parameter=K>V</parameter></function>."""\n    m = FUNC.search(c or "")\n    if not m:\n        return ""\n    name, p = m.group(1), dict(PARAM.findall(m.group(2)))\n    if name == "bash":\n        return "bash " + p.get("command", "")\n    if name == "str_replace_editor" and "command" in p:\n        cmd, path = p.pop("command"), p.pop("path", "")\n        return ("str_replace_editor %s %s %s" % (cmd, path, json.dumps(p, ensure_ascii=False) if p else "")).strip()\n    return "%s %s" % (name, json.dumps(p, ensure_ascii=False))\n\n\ndef _act(m):\n    if isinstance(m.get("action"), str) and m["action"].strip():\n        return m["action"].strip()\n    tc = m.get("tool_calls") or []\n    if tc:\n        f = tc[0].get("function", tc[0])\n        return "%s %s" % (f.get("name"), f.get("arguments"))\n    return _xml_act(_txt(m))\n\n\ndef _txt(m):\n    c = m.get("content")\n    return c if isinstance(c, str) else json.dumps(c)\n\n\nTEST = re.compile(r"pytest|unittest|\x08tox\x08|nosetests|manage\\.py test|runtests")\nFAILED = re.compile(r"(\\d+) failed")\nEDIT_A = re.compile(r"str_replace_editor\\s+(str_replace|create|insert)\\s+(\\S+)")\nEDIT_CMD = re.compile(r\'"command":\\s*"(str_replace|create|insert)"\')\nEDIT_PATH = re.compile(r\'"path":\\s*"([^"]+)"\')\n\n\ndef _tail2(p):\n    return "/".join(p.strip().split("/")[-2:])\n\n\ndef _edit_path(a):\n    m = EDIT_A.search(a)\n    if m:\n        return _tail2(m.group(2))\n    if EDIT_CMD.search(a):\n        q = EDIT_PATH.search(a)\n        return _tail2(q.group(1)) if q else None\n    return None\n\n\ndef patch_files(patch):\n    """File names touched by a unified diff, as last-two-path-component tails."""\n    return {_tail2(x) for x in re.findall(r"^diff --git a/(\\S+)", patch or "", re.M)}\n\n\ndef steps_of(messages):\n    """Raw chat messages (OpenHands or SWE-smith tool format) -> [{a, err, rev, edit, test, tfail}]; one entry per assistant action. Unparseable input -> None."""\n    msgs = _jl(messages)\n    if not isinstance(msgs, list):\n        return None\n    out = []\n    for i, m in enumerate(msgs):\n        if m.get("role") != "assistant":\n            continue\n        a = _act(m)\n        if not a:\n            continue\n        o = next((_txt(x) for x in msgs[i + 1:i + 3] if x.get("role") in ("tool", "user")), "")\n        is_test = bool(TEST.search(a))\n        f = FAILED.findall(o[-3000:]) if is_test else []\n        out.append(dict(a=a[:ACT_CHARS], paths=sorted({_tail2(x) for x in PATH.findall(o[:4000] + " " + a)})[:20], err=bool(ERR.search(o[:3000])), rev=bool(REV.search(a)), edit=_edit_path(a), test=is_test, tfail=(int(f[-1]) if f else (0 if is_test and re.search(r"\\d+ passed", o[-3000:]) else None))))\n    return out\n\n\ndef signal_counts(groups):\n    """groups: {instance: [{resolved, steps, patch_files}]}. Per step-level signal: [hits, steps] in resolved and in unresolved runs.\n    A signal is only informative if its rate differs between the two. S3 needs a resolved run of the same instance (its patch files = known right location)."""\n    sig = defaultdict(lambda: {"resolved": [0, 0], "unresolved": [0, 0]})\n    for runs in groups.values():\n        good = set().union(*[r.get("patch_files") or set() for r in runs if r["resolved"]]) if any(r["resolved"] for r in runs) else None\n        for r in runs:\n            k = "resolved" if r["resolved"] else "unresolved"\n            seen = Counter(); edits = Counter(); prev_fail = None\n            for st in r["steps"]:\n                hits = {"S1_error_after": st["err"], "S2_revert": st["rev"], "S6_action_repeated": seen[st["a"]] >= 1}\n                seen[st["a"]] += 1\n                if st["edit"]:\n                    edits[st["edit"]] += 1\n                    hits["S5_same_file_edited_3rd_time"] = edits[st["edit"]] == 3\n                    if good is not None:\n                        hits["S3_edit_outside_resolved_patch_files"] = st["edit"] not in good\n                    if r.get("patch_files") is not None:\n                        hits["S4_edit_not_in_own_final_patch"] = st["edit"] not in r["patch_files"]\n                if st["test"] and st["tfail"] is not None:\n                    hits["S7_test_run_failures_up"] = st["tfail"] > 0 and (prev_fail is None or st["tfail"] > prev_fail)\n                    hits["S8_test_run_all_pass"] = st["tfail"] == 0\n                    prev_fail = st["tfail"]\n                for n, v in hits.items():\n                    sig[n][k][1] += 1\n                    sig[n][k][0] += bool(v)\n    out = {}\n    for n, d in sig.items():\n        r_, u_ = d["resolved"], d["unresolved"]\n        rr, ur = r_[0] / max(r_[1], 1), u_[0] / max(u_[1], 1)\n        out[n] = dict(resolved=r_, unresolved=u_, rate_resolved=round(rr, 4), rate_unresolved=round(ur, 4), lift=(round(ur / rr, 2) if rr else None))\n    return out\n\n\ndef next_edit_stats(groups):\n    """B\': natural candidate sets. For each edit step of a RESOLVED run whose file is in its own final patch, candidates = file paths seen in earlier actions/observations of that run.\n    Gold = the edited file. Other candidates are real files with unknown relevance (never called wrong). Returns counts and set-size quantiles."""\n    n = cov = 0\n    sizes = []\n    for runs in groups.values():\n        for r in runs:\n            if not r["resolved"] or not r.get("patch_files"):\n                continue\n            seen = set()\n            for st in r["steps"]:\n                if st["edit"] and st["edit"] in r["patch_files"]:\n                    n += 1\n                    cov += st["edit"] in seen\n                    if st["edit"] in seen:\n                        sizes.append(len(seen))\n                seen.update(st.get("paths") or [])\n    sizes.sort()\n    q = lambda f: sizes[int(f * (len(sizes) - 1))] if sizes else None\n    return dict(edit_decisions=n, gold_in_candidates=cov, set_size_q10_q50_q90=[q(0.1), q(0.5), q(0.9)])\n\n\ndef build_rows(group, match="exact"):\n    """group: runs of one instance, [{tid, resolved, steps:[{a, err, rev}]}]. Returns decision rows (see module doc)."""\n    rows = []\n    seen_nodes = set()\n\n    def walk(runs, d, prefix):\n        if len(runs) < 2:\n            return\n        live = [r for r in runs if len(r["steps"]) > d]\n        by_a = defaultdict(list)\n        for r in live:\n            by_a[r["steps"][d]["a"] if match == "exact" else norm(r["steps"][d]["a"])].append(r)\n        if len(by_a) > 1 and (prefix, d) not in seen_nodes:\n            seen_nodes.add((prefix, d))\n            rows.extend(node_rows(by_a, d))\n        for a, rs in by_a.items():\n            walk(rs, d + 1, prefix + (a,))\n\n    def node_rows(by_a, d):\n        out = []\n        for g_key, g_runs in by_a.items():\n            gold_runs = [r for r in g_runs if r["resolved"]]\n            if not gold_runs:\n                continue\n            g = gold_runs[0]\n            g_act = g["steps"][d]["a"]\n            opts = [dict(text=g_act[:ACT_CHARS], tier="GOLD", executed_here=True, provenance=dict(tid=g["tid"], step=d, run_resolved=True, model=g.get("model")))]\n            for key, rs in by_a.items():\n                if rs is g_runs:\n                    continue\n                a = rs[0]["steps"][d]["a"]\n                res = [r for r in rs if r["resolved"]]\n                if res:\n                    opts.append(dict(text=a[:ACT_CHARS], tier="T3", executed_here=True, provenance=dict(tid=res[0]["tid"], step=d, run_resolved=True, kind="resolved_branch", model=res[0].get("model"))))\n                    continue\n                u = rs[0]\n                s = u["steps"][d]; nxt = u["steps"][d + 1] if d + 1 < len(u["steps"]) else None\n                signs = [k for k, v in (("error", s["err"]), ("revert", s["rev"]), ("retry", nxt is not None and nxt["a"] == s["a"])) if v]\n                opts.append(dict(text=a[:ACT_CHARS], tier="T1" if signs else "T2", executed_here=True,\n                                 provenance=dict(tid=u["tid"], step=d, run_resolved=False, n_failing_runs=len(rs), signs=signs, model=u.get("model"))))\n            texts = {o["text"] for o in opts}\n            for k in (d - 2, d - 1, d + 1, d + 2):\n                if 0 <= k < len(g["steps"]) and g["steps"][k]["a"][:ACT_CHARS] not in texts:\n                    texts.add(g["steps"][k]["a"][:ACT_CHARS])\n                    opts.append(dict(text=g["steps"][k]["a"][:ACT_CHARS], tier="T3", executed_here=False, provenance=dict(tid=g["tid"], step=k, run_resolved=True, kind="same_traj_nearby")))\n            for o in opts:\n                o["near_duplicate_of_gold"] = o["tier"] != "GOLD" and norm(o["text"]) == norm(g_act)\n            w = [TIER_W[o["tier"]] for o in opts]\n            for o, x in zip(opts, w):\n                o["soft_target"] = round(x / sum(w), 4)\n            out.append(dict(decision_id="%s:%d:%s" % (g["tid"], d, hashlib.sha1(g_act.encode()).hexdigest()[:8]), match=match, instance=g.get("instance"), state_ref=dict(tid=g["tid"], step=d), options=opts))\n        return out\n\n    walk(group, 0, ())\n    return rows\n\n\ndef summarise(rows):\n    c = Counter(); depth = []\n    for r in rows:\n        tiers = Counter(o["tier"] for o in r["options"] if not o["near_duplicate_of_gold"])\n        c["rows"] += 1\n        c["rows_with_T1"] += tiers["T1"] > 0\n        c["rows_with_T2_no_T1"] += tiers["T2"] > 0 and not tiers["T1"]\n        c["rows_T3_only"] += not tiers["T1"] and not tiers["T2"]\n        c["rows_at_step0"] += r["state_ref"]["step"] == 0\n        for o in r["options"]:\n            c["options_" + o["tier"] + ("_near_dup" if o["near_duplicate_of_gold"] else "")] += 1\n        depth.append(r["state_ref"]["step"])\n    c["depth_median"] = sorted(depth)[len(depth) // 2] if depth else None\n    return dict(c)\n\n\ndef selftest():\n    S = lambda *a: [dict(a=x, err=e, rev=False) for x, e in a]\n    group = [dict(tid="r1", resolved=True, instance="i", steps=S(("ls", 0), ("edit a.py", 0), ("run t", 0))),\n             dict(tid="u1", resolved=False, instance="i", steps=S(("ls", 0), ("edit b.py", 1), ("edit b.py", 1))),\n             dict(tid="u2", resolved=False, instance="i", steps=S(("ls", 0), ("cat c.py", 0), ("x", 0))),\n             dict(tid="r2", resolved=True, instance="i", steps=S(("pwd", 0), ("y", 0)))]\n    rows = build_rows(group)\n    assert any(r["state_ref"]["step"] == 0 for r in rows), "root divergence ls vs pwd must give a state"\n    n1 = [r for r in rows if r["state_ref"]["step"] == 1]\n    assert len(n1) == 1 and n1[0]["options"][0]["tier"] == "GOLD"\n    tiers = {o["tier"]: o for o in n1[0]["options"]}\n    assert tiers["T1"]["provenance"]["tid"] == "u1" and "error" in tiers["T1"]["provenance"]["signs"] and tiers["T1"]["executed_here"]\n    assert tiers["T2"]["provenance"]["tid"] == "u2"\n    assert any(o["tier"] == "T3" and not o["executed_here"] for o in n1[0]["options"])\n    assert abs(sum(o["soft_target"] for o in n1[0]["options"]) - 1) < 1e-3\n    raw = [{"role": "user", "content": "t"}, {"role": "assistant", "content": "", "action": "ls"}, {"role": "tool", "content": "Traceback (most recent call last)"}, {"role": "assistant", "content": "", "tool_calls": [{"function": {"name": "bash", "arguments": "{}"}}]}]\n    st = steps_of(raw)\n    assert [x["a"] for x in st] == ["ls", "bash {}"] and st[0]["err"] and steps_of("not json") is None\n    G = {"i": [dict(resolved=True, patch_files={"pkg/a.py"}, steps=[dict(a="x", err=False, rev=False, edit="pkg/a.py", test=False, tfail=None), dict(a="pytest", err=True, rev=False, edit=None, test=True, tfail=0)]),\n               dict(resolved=False, patch_files={"pkg/b.py"}, steps=[dict(a="y", err=False, rev=False, edit="pkg/b.py", test=False, tfail=None), dict(a="pytest", err=True, rev=False, edit=None, test=True, tfail=2)])]}\n    sc = signal_counts(G)\n    assert sc["S3_edit_outside_resolved_patch_files"]["unresolved"] == [1, 1] and sc["S3_edit_outside_resolved_patch_files"]["resolved"] == [0, 1]\n    assert sc["S7_test_run_failures_up"]["unresolved"] == [1, 1] and sc["S8_test_run_all_pass"]["resolved"] == [1, 1]\n    e = steps_of([{"role": "assistant", "content": "", "action": "str_replace_editor str_replace /testbed/pkg/a.py --old x"}, {"role": "tool", "content": ""}, {"role": "assistant", "content": "", "action": "python -m pytest t.py"}, {"role": "tool", "content": "== 3 failed, 4 passed =="}])\n    assert e[0]["edit"] == "pkg/a.py" and e[1]["test"] and e[1]["tfail"] == 3\n    assert patch_files("diff --git a/src/x/y.py b/src/x/y.py\\n") == {"x/y.py"}\n    x = steps_of([{"role": "user", "content": "task"}, {"role": "assistant", "content": "Let me look.\\n\\n<function=str_replace_editor>\\n<parameter=command>view</parameter>\\n<parameter=path>/testbed/pkg/a.py</parameter>\\n</function>"}, {"role": "user", "content": "OBS"}, {"role": "assistant", "content": "<function=bash>\\n<parameter=command>python -m pytest t.py</parameter>\\n</function>"}, {"role": "user", "content": "1 failed"}])\n    assert len(x) == 2 and x[0]["a"].startswith("str_replace_editor view /testbed/pkg/a.py") and x[1]["a"] == "bash python -m pytest t.py" and x[1]["test"] and x[1]["tfail"] == 1\n    ne = next_edit_stats({"i": [dict(resolved=True, patch_files={"pkg/a.py"}, steps=[dict(a="x", paths=["pkg/a.py", "pkg/b.py"], err=False, rev=False, edit=None, test=False, tfail=None), dict(a="e", paths=[], err=False, rev=False, edit="pkg/a.py", test=False, tfail=None)])]})\n    assert ne == dict(edit_decisions=1, gold_in_candidates=1, set_size_q10_q50_q90=[2, 2, 2]), ne\n    nm = build_rows(group, match="norm")\n    assert all(r["match"] == "norm" for r in nm)\n    print("selftest ok", summarise(rows))\n\n\nif __name__ == "__main__":\n    if sys.argv[1] == "selftest":\n        selftest()\n    else:\n        runs = [json.loads(l) for l in open(sys.argv[2], encoding="utf-8")]\n        by = defaultdict(lambda: defaultdict(list))\n        for r in runs:\n            by[r["src"]][r["instance"]].append(dict(tid=r["tid"], resolved=r["resolved"], steps=r["steps"], instance=r["instance"]))\n        print(json.dumps({s: dict(instances=len(g), instances_mixed=sum(len({x["resolved"] for x in v}) == 2 for v in g.values()),\n                                  **summarise([row for v in g.values() for row in build_rows(v)])) for s, g in by.items()}, indent=1))\n'
open('/kaggle/working/hybrid_options.py', 'w', encoding='utf-8').write(SRC)
import sys; sys.path.insert(0, '/kaggle/working')
import hybrid_options as H
H.selftest()

In [ ]:
import os, sys, json, glob, gzip, time, subprocess, fnmatch
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "pyarrow"], check=True)
import pyarrow.parquet as pq
from collections import defaultdict, Counter
from huggingface_hub import HfApi, snapshot_download
api = HfApi()
SRC_DS = {
    "nebius":        dict(repo="nebius/SWE-rebench-openhands-trajectories", pat=["*.parquet"], need={"trajectory_id", "instance_id", "repo", "trajectory", "resolved", "model_patch"}, msgs="trajectory", tid="trajectory_id", patch="model_patch"),
    "swesmith_tool": dict(repo="SWE-bench/SWE-smith-trajectories", pat=["data/tool-*.parquet"], need={"traj_id", "instance_id", "messages", "resolved", "model", "patch"}, msgs="messages", tid="traj_id", patch="patch"),
}
for k, s in SRC_DS.items():     # fail fast on schema before the big download
    s["rev"] = api.dataset_info(s["repo"]).sha
    names = [f for f in api.list_repo_files(s["repo"], repo_type="dataset", revision=s["rev"]) if any(fnmatch.fnmatch(f, p) for p in s["pat"])]
    from huggingface_hub import hf_hub_download
    cols = pq.ParquetFile(hf_hub_download(s["repo"], names[0], repo_type="dataset", revision=s["rev"])).schema_arrow.names
    print(k, s["rev"][:10], len(names), "files", cols)
    assert s["need"] <= set(cols), (k, s["need"] - set(cols))
    s["path"] = snapshot_download(s["repo"], repo_type="dataset", revision=s["rev"], allow_patterns=s["pat"])

In [ ]:
t0 = time.time(); groups = {}; skipped = Counter(); n_traj = Counter()
for k, s in SRC_DS.items():
    g = defaultdict(list)
    for f in sorted(glob.glob(s["path"] + "/**/*.parquet", recursive=True)):
        cols = [s["tid"], "instance_id", s["msgs"], "resolved"] + [s["patch"]] + [c for c in ("repo", "model") if c in s["need"]]
        for b in pq.ParquetFile(f).iter_batches(batch_size=64, columns=cols):
            for r in b.to_pylist():
                st = H.steps_of(r[s["msgs"]])
                if not st:
                    skipped[k] += 1; continue
                n_traj[k] += 1
                g[r["instance_id"]].append(dict(tid=str(r[s["tid"]]), resolved=bool(r["resolved"]), steps=st, instance=r["instance_id"], repo=r.get("repo"), model=r.get("model"), patch_files=H.patch_files(r[s["patch"]])))
    groups[k] = g
    print(k, "trajectories", n_traj[k], "instances", len(g), "skipped", skipped[k], "| %.0f s" % (time.time() - t0), flush=True)
scan_seconds = time.time() - t0

In [ ]:
os.makedirs("/kaggle/working/out", exist_ok=True)
counts, sigs, NEXT_EDIT = {}, {}, {}
t1 = time.time()
with gzip.open("/kaggle/working/out/rows_hybrid.jsonl.gz", "wt", encoding="utf-8") as fo:
    for k, g in groups.items():
        sigs[k] = H.signal_counts(g)
        next_edit = NEXT_EDIT[k] = H.next_edit_stats(g); print(k, 'next-edit candidate sets', json.dumps(next_edit), flush=True)
        print(k, "signals", json.dumps({n: (v["rate_resolved"], v["rate_unresolved"], v["lift"]) for n, v in sigs[k].items()}), flush=True)
        for mode in ("exact", "norm"):
            rows = []
            for iid, runs in g.items():
                for r in H.build_rows(runs, match=mode):
                    r["source"] = k; r["repo"] = runs[0].get("repo") or (iid.split(".")[0].replace("__", "/", 1)); rows.append(r)
                    fo.write(json.dumps(r, ensure_ascii=False) + "\n")
            real = [(r, o) for r in rows for o in r["options"] if o["tier"] in ("T1", "T2") and not o["near_duplicate_of_gold"]]
            same_model = sum(1 for r, o in real if o["provenance"].get("model") and o["provenance"]["model"] == r["options"][0]["provenance"].get("model"))
            signs = Counter(sg for r, o in real if o["tier"] == "T1" for sg in o["provenance"]["signs"])
            tl = defaultdict(list)
            for r in rows:
                for o in r["options"]: tl[o["tier"]].append(len(o["text"]))
            mult = Counter(len(v) for v in g.values())
            counts[k + ":" + mode] = dict(trajectories=n_traj[k], instances=len(g), instances_multi=sum(len(v) > 1 for v in g.values()),
                                          instances_mixed=sum(len({x["resolved"] for x in v}) == 2 for v in g.values()), runs_per_instance_hist=dict(sorted(mult.items())[:12]),
                                          skipped_unparseable=skipped[k], real_options=len(real), real_option_same_generator_model=same_model, T1_signs=dict(signs),
                                          mean_chars_by_tier={x: round(sum(v) / len(v), 1) for x, v in tl.items()},
                                          rows_with_real_option=len({r["decision_id"] for r, o in real}), repos_with_real_rows=len({r["repo"] for r, o in real}),
                                          real_rows_by_depth=dict(Counter(min(r["state_ref"]["step"], 10) for r in {x["decision_id"]: x for x, _ in real}.values())), real_rows_by_tier_and_depth=dict(Counter(o["tier"] + ":" + ("d0" if r["state_ref"]["step"] == 0 else "d1-4" if r["state_ref"]["step"] < 5 else "d5+") for r, o in real)), **H.summarise(rows))
            print(k, mode, json.dumps(counts[k + ":" + mode]), flush=True)
json.dump(dict(label="exploratory (no gate)", date=time.strftime("%Y%m%d-%H%M%S"), counts=counts, step_signals=sigs, next_edit_sets=NEXT_EDIT, scan_seconds=round(scan_seconds, 1), build_seconds=round(time.time() - t1, 1),
               revisions={k: s["rev"] for k, s in SRC_DS.items()}, tier_weights=H.TIER_W,
               caveats=["exact = identical earlier ACTION text; norm = identical earlier (tool, subcommand, file) triples; observations are not compared in either", "T1/T2 flags and S1-S8 come from weak regexes; an error after an action does not prove it caused the failure", "near_duplicate_of_gold options are excluded from the real-wrong counts", "soft_target is a fixed evidence prior, not an outcome", "a signal separates failing from passing runs only if its rate differs (lift far from 1)"]),
          open("/kaggle/working/out/counts.json", "x"), indent=1)
print("wrote counts.json")